In [1]:
# ============================================================
# Task 5: Decision Trees and Random Forests
# Heart Disease Dataset
# ElevateLabs AI/ML Internship
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150

# Feature names for readability
FEATURE_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal"
]
FEATURE_LABELS = {
    "age":      "Age",
    "sex":      "Sex",
    "cp":       "Chest Pain Type",
    "trestbps": "Resting BP",
    "chol":     "Cholesterol",
    "fbs":      "Fasting Blood Sugar",
    "restecg":  "Rest ECG",
    "thalach":  "Max Heart Rate",
    "exang":    "Exercise Angina",
    "oldpeak":  "ST Depression",
    "slope":    "Slope",
    "ca":       "Major Vessels",
    "thal":     "Thal"
}


In [2]:
# ============================================================
# STEP 1 — Load & Explore
# ============================================================
print("=" * 60)
print("STEP 1: Load & Explore Dataset")
print("=" * 60)

df = pd.read_csv("heart.csv")
print(f"\n📌 Shape: {df.shape}")
print(f"\n📌 Target Distribution:")
print(df["target"].value_counts().rename({0: "No Disease", 1: "Heart Disease"}))
print(f"\n📌 Missing Values: {df.isnull().sum().sum()}")
print(f"\n📌 Class Balance: {df['target'].mean()*100:.1f}% Heart Disease")


STEP 1: Load & Explore Dataset

📌 Shape: (1025, 14)

📌 Target Distribution:
target
Heart Disease    526
No Disease       499
Name: count, dtype: int64

📌 Missing Values: 0

📌 Class Balance: 51.3% Heart Disease


In [3]:
# ============================================================
# STEP 2 — Preprocessing
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Preprocessing")
print("=" * 60)

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"✅ Stratified split applied (preserves class ratio)")



STEP 2: Preprocessing
✅ Train: 820 | Test: 205
✅ Stratified split applied (preserves class ratio)


In [4]:
# ============================================================
# STEP 3 — Decision Tree: Depth Analysis (Overfitting)
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Decision Tree — Depth Analysis (Overfitting Study)")
print("=" * 60)

depths      = list(range(1, 16))
train_accs  = []
test_accs   = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt.predict(X_train)))
    test_accs.append(accuracy_score(y_test, dt.predict(X_test)))

best_depth = depths[np.argmax(test_accs)]
print(f"\n📌 Best depth by test accuracy: {best_depth} → {max(test_accs)*100:.2f}%")



STEP 3: Decision Tree — Depth Analysis (Overfitting Study)

📌 Best depth by test accuracy: 9 → 98.54%


In [5]:
# ============================================================
# STEP 4 — Train Final Decision Tree (best depth)
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Train Decision Tree (max_depth={})".format(best_depth))
print("=" * 60)

dt_best = DecisionTreeClassifier(max_depth=best_depth, random_state=42, criterion="entropy")
dt_best.fit(X_train, y_train)
y_pred_dt = dt_best.predict(X_test)

acc_dt = accuracy_score(y_test, y_pred_dt)
print(f"\n  Accuracy : {acc_dt*100:.2f}%")
print(f"\n📌 Classification Report (Decision Tree):")
print(classification_report(y_test, y_pred_dt, target_names=["No Disease", "Heart Disease"]))

# Text representation of tree (first 3 levels)
print("\n📌 Decision Tree Rules (top 3 levels):")
tree_rules = export_text(dt_best, feature_names=FEATURE_NAMES, max_depth=3)
print(tree_rules)


STEP 4: Train Decision Tree (max_depth=9)

  Accuracy : 100.00%

📌 Classification Report (Decision Tree):
               precision    recall  f1-score   support

   No Disease       1.00      1.00      1.00       100
Heart Disease       1.00      1.00      1.00       105

     accuracy                           1.00       205
    macro avg       1.00      1.00      1.00       205
 weighted avg       1.00      1.00      1.00       205


📌 Decision Tree Rules (top 3 levels):
|--- cp <= 0.50
|   |--- ca <= 0.50
|   |   |--- thal <= 2.50
|   |   |   |--- exang <= 0.50
|   |   |   |   |--- truncated branch of depth 5
|   |   |   |--- exang >  0.50
|   |   |   |   |--- truncated branch of depth 4
|   |   |--- thal >  2.50
|   |   |   |--- oldpeak <= 0.65
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- oldpeak >  0.65
|   |   |   |   |--- class: 0
|   |--- ca >  0.50
|   |   |--- oldpeak <= 0.45
|   |   |   |--- thalach <= 159.50
|   |   |   |   |--- truncated branch of dep

In [6]:
# ============================================================
# STEP 5 — Train Random Forest
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Train Random Forest")
print("=" * 60)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"\n  Accuracy : {acc_rf*100:.2f}%")
print(f"\n📌 Classification Report (Random Forest):")
print(classification_report(y_test, y_pred_rf, target_names=["No Disease", "Heart Disease"]))


STEP 5: Train Random Forest

  Accuracy : 100.00%

📌 Classification Report (Random Forest):
               precision    recall  f1-score   support

   No Disease       1.00      1.00      1.00       100
Heart Disease       1.00      1.00      1.00       105

     accuracy                           1.00       205
    macro avg       1.00      1.00      1.00       205
 weighted avg       1.00      1.00      1.00       205



In [7]:
# ============================================================
# STEP 6 — Cross-Validation
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: Cross-Validation (5-Fold Stratified)")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_dt = cross_val_score(dt_best, X, y, cv=cv, scoring="accuracy")
cv_rf = cross_val_score(rf,      X, y, cv=cv, scoring="accuracy")

print(f"\n  Decision Tree CV: {cv_dt.round(4)} | Mean: {cv_dt.mean()*100:.2f}% ± {cv_dt.std()*100:.2f}%")
print(f"  Random Forest CV: {cv_rf.round(4)} | Mean: {cv_rf.mean()*100:.2f}% ± {cv_rf.std()*100:.2f}%")



STEP 6: Cross-Validation (5-Fold Stratified)

  Decision Tree CV: [1.     0.9902 1.     0.9659 0.9951] | Mean: 99.02% ± 1.27%
  Random Forest CV: [1.     1.     1.     0.9805 1.    ] | Mean: 99.61% ± 0.78%


In [8]:
# ============================================================
# STEP 7 — Feature Importances
# ============================================================
print("\n" + "=" * 60)
print("STEP 7: Feature Importances")
print("=" * 60)

fi_dt = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "DT Importance": dt_best.feature_importances_,
    "RF Importance": rf.feature_importances_
}).sort_values("RF Importance", ascending=False)

print("\n📌 Feature Importances (sorted by Random Forest):")
print(fi_dt.to_string(index=False))



STEP 7: Feature Importances

📌 Feature Importances (sorted by Random Forest):
 Feature  DT Importance  RF Importance
      cp       0.230380       0.142094
 thalach       0.043755       0.117349
      ca       0.122085       0.114844
 oldpeak       0.098915       0.112634
    thal       0.093638       0.095930
     age       0.118180       0.091285
    chol       0.099915       0.077771
   exang       0.023441       0.073707
trestbps       0.053638       0.067765
   slope       0.015245       0.048711
     sex       0.051162       0.026682
 restecg       0.049646       0.020438
     fbs       0.000000       0.010790


In [9]:
# ============================================================
# ===  ALL PLOTS  ============================================
# ============================================================
print("\n" + "=" * 60)
print("STEP 8: Generating Plots")
print("=" * 60)

# ── PLOT 1: Decision Tree Visualization ──────────────────────
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_best, feature_names=FEATURE_NAMES,
          class_names=["No Disease", "Heart Disease"],
          filled=True, rounded=True, fontsize=9, ax=ax,
          impurity=True, proportion=False)
plt.title(f"Decision Tree (max_depth={best_depth}, criterion=entropy)",
          fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("dt_01_tree_visualization.png", bbox_inches="tight")
plt.close()
print("✅ Saved: dt_01_tree_visualization.png")



STEP 8: Generating Plots
✅ Saved: dt_01_tree_visualization.png


In [10]:
# ── PLOT 2: Overfitting Analysis ─────────────────────────────
plt.figure(figsize=(10, 6))
plt.plot(depths, [a*100 for a in train_accs], "o-", color="#3498db",
         linewidth=2, markersize=6, label="Train Accuracy")
plt.plot(depths, [a*100 for a in test_accs],  "s-", color="#e74c3c",
         linewidth=2, markersize=6, label="Test Accuracy")
plt.axvline(best_depth, color="#2ecc71", linestyle="--",
            linewidth=2, label=f"Best depth = {best_depth}")
plt.fill_between(depths,
                 [t*100 for t in train_accs],
                 [t*100 for t in test_accs],
                 alpha=0.1, color="#e74c3c", label="Overfitting gap")
plt.xlabel("Tree Depth (max_depth)")
plt.ylabel("Accuracy (%)")
plt.title("Decision Tree: Overfitting Analysis\n(Train vs Test Accuracy by Depth)",
          fontsize=13, fontweight="bold")
plt.legend(fontsize=10)
plt.xticks(depths)
plt.tight_layout()
plt.savefig("dt_02_overfitting_analysis.png", bbox_inches="tight")
plt.close()
print("✅ Saved: dt_02_overfitting_analysis.png")


✅ Saved: dt_02_overfitting_analysis.png


In [11]:
# ── PLOT 3: Confusion Matrices (DT vs RF) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (y_p, title, acc_val) in zip(axes, [
    (y_pred_dt, f"Decision Tree (depth={best_depth})\nAccuracy: {acc_dt*100:.2f}%", acc_dt),
    (y_pred_rf, f"Random Forest (100 trees)\nAccuracy: {acc_rf*100:.2f}%", acc_rf)
]):
    cm = confusion_matrix(y_test, y_p)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["No Disease", "Heart Disease"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title, fontsize=12, fontweight="bold")

plt.suptitle("Confusion Matrices — Decision Tree vs Random Forest",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("dt_03_confusion_matrices.png", bbox_inches="tight")
plt.close()
print("✅ Saved: dt_03_confusion_matrices.png")

✅ Saved: dt_03_confusion_matrices.png


In [12]:
# ── PLOT 4: Feature Importances (DT vs RF side by side) ──────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

fi_dt_sorted = fi_dt.sort_values("DT Importance", ascending=True)
fi_rf_sorted = fi_dt.sort_values("RF Importance", ascending=True)

axes[0].barh(fi_dt_sorted["Feature"], fi_dt_sorted["DT Importance"],
             color="#3498db", edgecolor="white", alpha=0.85)
axes[0].set_title(f"Decision Tree\nFeature Importances", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Importance (Gini/Entropy reduction)")

axes[1].barh(fi_rf_sorted["Feature"], fi_rf_sorted["RF Importance"],
             color="#e74c3c", edgecolor="white", alpha=0.85)
axes[1].set_title("Random Forest\nFeature Importances", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Importance (Mean decrease in impurity)")

plt.suptitle("Feature Importances — Decision Tree vs Random Forest",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("dt_04_feature_importances.png", bbox_inches="tight")
plt.close()
print("✅ Saved: dt_04_feature_importances.png")

✅ Saved: dt_04_feature_importances.png


In [13]:
# ── PLOT 5: Cross-Validation Scores ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(5)
w = 0.35
fold_labels = [f"Fold {i+1}" for i in range(5)]

bars1 = ax.bar(x - w/2, cv_dt * 100, w, label=f"Decision Tree (mean={cv_dt.mean()*100:.2f}%)",
               color="#3498db", alpha=0.85, edgecolor="white")
bars2 = ax.bar(x + w/2, cv_rf * 100, w, label=f"Random Forest (mean={cv_rf.mean()*100:.2f}%)",
               color="#e74c3c", alpha=0.85, edgecolor="white")

ax.axhline(cv_dt.mean() * 100, color="#3498db", linestyle="--", linewidth=1.5, alpha=0.7)
ax.axhline(cv_rf.mean() * 100, color="#e74c3c", linestyle="--", linewidth=1.5, alpha=0.7)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}%", ha="center", fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(fold_labels)
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(70, 105)
ax.set_title("5-Fold Cross-Validation Accuracy\nDecision Tree vs Random Forest",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("dt_05_cross_validation.png", bbox_inches="tight")
plt.close()
print("✅ Saved: dt_05_cross_validation.png")

✅ Saved: dt_05_cross_validation.png


In [14]:
# ── PLOT 6: Model Comparison Summary ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Accuracy bar: single test + CV mean
models      = ["Decision Tree\n(test)", "Random Forest\n(test)",
                "Decision Tree\n(CV mean)", "Random Forest\n(CV mean)"]
accuracies  = [acc_dt*100, acc_rf*100, cv_dt.mean()*100, cv_rf.mean()*100]
colors      = ["#3498db", "#e74c3c", "#2980b9", "#c0392b"]
bars = axes[0].bar(models, accuracies, color=colors, edgecolor="white", alpha=0.85)
axes[0].set_ylim(70, 105)
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Accuracy Comparison", fontsize=12, fontweight="bold")
for bar, val in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val:.2f}%", ha="center", fontweight="bold", fontsize=10)

# Entropy concept illustration
z = np.linspace(0.001, 0.999, 300)
entropy = -z * np.log2(z) - (1-z) * np.log2(1-z)
gini    = 2 * z * (1 - z)

axes[1].plot(z, entropy, color="#9b59b6", linewidth=2.5, label="Entropy = −p·log₂(p) − (1−p)·log₂(1−p)")
axes[1].plot(z, gini,    color="#e67e22", linewidth=2.5, label="Gini = 2p(1−p)")
axes[1].axvline(0.5, color="gray", linestyle="--", linewidth=1.2, label="Max impurity (p=0.5)")
axes[1].set_xlabel("p (probability of class 1)")
axes[1].set_ylabel("Impurity")
axes[1].set_title("Entropy vs Gini Impurity", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9)

plt.suptitle("Model Comparison & Impurity Metrics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("dt_06_comparison_entropy.png", bbox_inches="tight")
plt.close()
print("✅ Saved: dt_06_comparison_entropy.png")

✅ Saved: dt_06_comparison_entropy.png


In [15]:
# ── PLOT 7: Number of Trees in RF vs Accuracy ────────────────
n_trees_range = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200]
rf_test_accs  = []
rf_cv_accs    = []

for n in n_trees_range:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    rf_test_accs.append(accuracy_score(y_test, rf_n.predict(X_test)) * 100)
    rf_cv_accs.append(cross_val_score(rf_n, X, y, cv=3, scoring="accuracy").mean() * 100)

plt.figure(figsize=(10, 6))
plt.plot(n_trees_range, rf_test_accs, "o-", color="#e74c3c",
         linewidth=2, markersize=7, label="Test Accuracy")
plt.plot(n_trees_range, rf_cv_accs,  "s-", color="#2ecc71",
         linewidth=2, markersize=7, label="CV Accuracy (3-fold)")
plt.axvline(100, color="gray", linestyle="--", linewidth=1.5, label="n_estimators=100")
plt.xlabel("Number of Trees (n_estimators)")
plt.ylabel("Accuracy (%)")
plt.title("Random Forest: Accuracy vs Number of Trees\n(More trees = more stable, diminishing returns)",
          fontsize=13, fontweight="bold")
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig("dt_07_rf_n_trees.png", bbox_inches="tight")
plt.close()
print("✅ Saved: dt_07_rf_n_trees.png")

✅ Saved: dt_07_rf_n_trees.png


In [16]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"""
┌────────────────────────────────────────────────────────────┐
│                    MODEL PERFORMANCE                       │
├──────────────────────┬──────────────────┬──────────────────┤
│ Metric               │ Decision Tree    │ Random Forest    │
├──────────────────────┼──────────────────┼──────────────────┤
│ Test Accuracy        │ {acc_dt*100:>10.2f}%    │ {acc_rf*100:>10.2f}%    │
│ CV Mean Accuracy     │ {cv_dt.mean()*100:>10.2f}%    │ {cv_rf.mean()*100:>10.2f}%    │
│ CV Std Dev           │ {cv_dt.std()*100:>10.2f}%    │ {cv_rf.std()*100:>10.2f}%    │
│ Tree Depth           │ {best_depth:>16}  │ {'100 trees (avg)':>16}  │
└──────────────────────┴──────────────────┴──────────────────┘
""")
print(f"  📌 Top feature (RF): '{fi_dt.iloc[0]['Feature']}' ({fi_dt.iloc[0]['RF Importance']:.4f})")
print(f"  📌 RF improved test accuracy by {(acc_rf - acc_dt)*100:.2f}% over single DT")
print(f"  📌 RF has lower CV std ({cv_rf.std()*100:.2f}%) → more stable (less variance)")
print("\n🎉 Task 5 Complete! All 7 plots saved.")



FINAL SUMMARY

┌────────────────────────────────────────────────────────────┐
│                    MODEL PERFORMANCE                       │
├──────────────────────┬──────────────────┬──────────────────┤
│ Metric               │ Decision Tree    │ Random Forest    │
├──────────────────────┼──────────────────┼──────────────────┤
│ Test Accuracy        │     100.00%    │     100.00%    │
│ CV Mean Accuracy     │      99.02%    │      99.61%    │
│ CV Std Dev           │       1.27%    │       0.78%    │
│ Tree Depth           │                9  │  100 trees (avg)  │
└──────────────────────┴──────────────────┴──────────────────┘

  📌 Top feature (RF): 'cp' (0.1421)
  📌 RF improved test accuracy by 0.00% over single DT
  📌 RF has lower CV std (0.78%) → more stable (less variance)

🎉 Task 5 Complete! All 7 plots saved.
